# Demo CPMK3 dengan foto baru

Jalankan notebook di sesi aktif sampai tombol unggah muncul. Unggah JPG yang baru diambil, pilih model, lalu jalankan prediksi. Geser ambang skor untuk melihat perubahan mask pada foto yang sama. Pilih kelas dan set data untuk melihat AP mask kelas itu.

Kalau tombol unggah belum muncul, tambahkan foto ke Input dan isi `DEMO_PATH` di sel terakhir. Saat tanya jawab, tabel AP per kelas, ringkasan mAP, dan gambar lima kasus bisa dipakai untuk menjelaskan hasil. [Hasil evaluasi CPMK3](https://github.com/cheskyy/cpmk3-instance-segmentation/blob/main/hasil/README.md)


In [ ]:
# Dependensi tambahan; PyTorch/CUDA bawaan Kaggle tidak diganti.
import os, sys, subprocess, importlib.util, importlib.metadata
os.environ.setdefault('YOLO_CONFIG_DIR', '/kaggle/working/yolo-settings' if os.path.isdir('/kaggle/working') else '/tmp/cpmk3-yolo-settings')
os.makedirs(os.environ['YOLO_CONFIG_DIR'], exist_ok=True)
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cpmk3-matplotlib')
required = {'ultralytics': 'ultralytics==8.3.228', 'pycocotools': 'pycocotools==2.0.11',
            'yaml': 'pyyaml', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
            'PIL': 'pillow', 'cv2': 'opencv-python-headless', 'scipy': 'scipy', 'IPython': 'ipython'}
install = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec('ultralytics') is not None and importlib.metadata.version('ultralytics') != '8.3.228':
    install.append('ultralytics==8.3.228')
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', *install])
import copy, csv, gc, hashlib, io, json, math, random, re, shutil, time, warnings, zipfile
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps, ExifTags
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops.misc import FrozenBatchNorm2d
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from IPython.display import display
# Tangkap konflik torch/torchvision sebelum berjam-jam training.
_ = torchvision.ops.nms(torch.tensor([[0., 0., 2., 2.]]), torch.tensor([1.]), 0.5)
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| ultralytics', importlib.metadata.version('ultralytics'))


In [ ]:
INPUT_ROOT = Path('/kaggle/input')
BASE_SIZE = 640
PRED_CONF = 0.001
PRED_MAX_DET = 100
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
base_candidates = []
for file in INPUT_ROOT.rglob('canonical_train.json'):
    work = file.parent.parent
    if all((work/'runs'/run).is_file() for run in (
        'yolo11n_640/weights/best.pt','yolo11s_640/weights/best.pt','maskrcnn_640/best.pth'
    )):
        base_candidates.append(file)
assert len(base_candidates)==1, 'Pasang output pelatihan awal yang memuat ketiga bobot 640.'
catalog_path=base_candidates[0]
catalog=json.loads(catalog_path.read_text())
CAT_NAMES={cat['id']:cat['name'] for cat in catalog['categories']}
CAT_IDS=sorted(CAT_NAMES)
CAT2INDEX={cid:i for i,cid in enumerate(CAT_IDS)}
INDEX2CAT={i:cid for cid,i in CAT2INDEX.items()}
NC=len(CAT_IDS)
TRAIN_BASE_FINGERPRINT=json.loads((catalog_path.parent/'audit_summary.json').read_text())['fingerprint']
RUNS=catalog_path.parent.parent/'runs'
BASELINES=[
    ('YOLO11n','yolo',RUNS/'yolo11n_640/weights/best.pt'),
    ('YOLO11s','yolo',RUNS/'yolo11s_640/weights/best.pt'),
    ('MaskRCNN','maskrcnn',RUNS/'maskrcnn_640/best.pth'),
]
metric_candidates=list(INPUT_ROOT.rglob('ringkasan_hasil.json'))
assert len(metric_candidates)==1,'Pasang output versi evaluasi lengkap.'
METRICS=metric_candidates[0].parent
print('Siap memuat bobot pada',DEVICE)


In [ ]:
def encode_mask(mask):
    r = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {'size': [int(x) for x in r['size']], 'counts': r['counts'].decode('ascii')}
def decode_seg(seg, h, w):
    if isinstance(seg, list):
        polygons = [p for p in seg if len(p) >= 6 and len(p) % 2 == 0]
        if len(polygons) != len(seg) or not polygons: raise ValueError('Polygon kosong/tidak valid')
        r = mask_utils.merge(mask_utils.frPyObjects(polygons, h, w))
    elif isinstance(seg, dict):
        if list(seg['size']) != [h, w]: raise ValueError('Ukuran RLE tidak cocok')
        r = mask_utils.frPyObjects(seg, h, w) if isinstance(seg['counts'], list) else {
            'size': seg['size'], 'counts': seg['counts'].encode('ascii') if isinstance(seg['counts'], str) else seg['counts']}
    else: raise ValueError('Format segmentation tidak dikenali')
    mask = mask_utils.decode(r)
    if mask.ndim == 3: mask = np.any(mask, axis=2)
    return mask.astype(np.uint8)

def build_maskrcnn(pretrained=False,size=BASE_SIZE):
    # Arsitektur normalisasi identik saat train dan load. Tidak mengunduh bobot saat evaluasi.
    from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
    fpn = resnet_fpn_backbone(backbone_name='resnet50',weights=None,
                             norm_layer=FrozenBatchNorm2d,trainable_layers=3)
    model = torchvision.models.detection.MaskRCNN(fpn,num_classes=91,
        min_size=size,max_size=size,box_detections_per_img=PRED_MAX_DET,
        box_score_thresh=PRED_CONF,box_nms_thresh=0.7)
    if pretrained:
        weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        state = weights.get_state_dict(progress=True,check_hash=True)
        model.load_state_dict(state,strict=True)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,NC+1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,NC+1)
    return model


def load_mask_model(path,size=None):
    # weights_only=True; checkpoint hanya tensor dan tipe Python dasar.
    ck = torch.load(path,map_location='cpu',weights_only=True)
    assert ck['fingerprint']==TRAIN_BASE_FINGERPRINT and ck['category_ids']==CAT_IDS,'Checkpoint berasal dari data/kelas berbeda.'
    model = build_maskrcnn(pretrained=False,size=size or ck['size'])
    model.load_state_dict(ck['state_dict'],strict=True)
    return model.to(DEVICE).eval()

def resize_longside(arr,size):
    h,w = arr.shape[:2];scale = size/max(h,w)
    nw,nh = max(1,round(w*scale)),max(1,round(h*scale))
    return cv2.resize(arr,(nw,nh),interpolation=cv2.INTER_LINEAR)

def model_predict_image(model,kind,arr,size):
    # Inferensi pada gambar berukuran terbatas, lalu setiap mask diproyeksi balik satu per satu.
    # Ini menghindari tensor 100 x 4080 x 2296 di GPU ketika foto HP beresolusi tinggi.
    small = resize_longside(arr,size);h,w = arr.shape[:2];result = []
    if kind=='yolo':
        res = model.predict(cv2.cvtColor(small,cv2.COLOR_RGB2BGR),imgsz=size,
            device=0 if DEVICE.type=='cuda' else 'cpu',conf=PRED_CONF,iou=0.7,
            max_det=PRED_MAX_DET,retina_masks=True,rect=False,verbose=False)[0]
        if res.masks is not None:
            for box,mask in zip(res.boxes,res.masks.data):
                idx = int(box.cls.item());score = float(box.conf.item())
                if idx not in INDEX2CAT: raise ValueError('Output kelas YOLO tidak cocok dengan mapping.')
                m = mask.detach().cpu().numpy()
                full = cv2.resize(m.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST)
                if full.any():result.append({'category_id':INDEX2CAT[idx],'segmentation':encode_mask(full),'score':score})
        del res
    else:
        with torch.inference_mode(): output = model([to_tensor(small).to(DEVICE)])[0]
        labels = output['labels'].cpu().tolist();scores = output['scores'].cpu().tolist()
        for k,(lab,score) in enumerate(zip(labels,scores)):
            if score<PRED_CONF:continue
            if lab-1 not in INDEX2CAT:raise ValueError('Output kelas Mask R-CNN tidak cocok.')
            m = output['masks'][k,0].detach().float().cpu().numpy()
            full = (cv2.resize(m,(w,h),interpolation=cv2.INTER_LINEAR)>=0.5).astype(np.uint8)
            if full.any():result.append({'category_id':INDEX2CAT[lab-1],'segmentation':encode_mask(full),'score':float(score)})
        del output
    return result


def load_run_model(kind,checkpoint,size):
    return YOLO(str(checkpoint)) if kind=='yolo' else load_mask_model(checkpoint,size)



In [ ]:
# Ubah tiga nilai ini jika memakai file input Kaggle saat sesi demo.
DEMO_PATH = None
DEMO_MODEL = 'YOLO11s'
DEMO_THRESHOLD = 0.35

def predict_demo(photo, model_name):
    '''Inferensi foto baru menggunakan checkpoint 640.'''
    if isinstance(photo,(bytes,bytearray,memoryview)):
        photo = io.BytesIO(bytes(photo))
    with Image.open(photo) as img:
        image = ImageOps.exif_transpose(img).convert('RGB')
        image.thumbnail((1920,1920))
        arr = np.asarray(image).copy()
    options = {name:(kind,ckpt) for name,kind,ckpt in BASELINES}
    kind,weights = options[model_name]
    predictor = load_run_model(kind,weights,BASE_SIZE)
    try:
        pred = model_predict_image(predictor,kind,arr,BASE_SIZE)
    finally:
        del predictor
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return arr,pred

def render_demo(arr, predictions, threshold, model_name):
    '''Ubah ambang skor dan gambar ulang mask dari prediksi foto yang sama.'''
    if not PRED_CONF <= threshold <= 1.0:
        raise ValueError(f'Gunakan ambang antara {PRED_CONF} dan 1.')
    overlay = arr.copy()
    h,w = arr.shape[:2]
    kept = [p for p in predictions if p['score']>=threshold]
    for index,p in enumerate(sorted(kept,key=lambda row:row['score'])):
        mask = decode_seg(p['segmentation'],h,w).astype(bool)
        if not mask.any():
            continue
        color = np.random.default_rng(index+41).integers(65,245,3)
        overlay[mask] = (0.56*overlay[mask] + 0.44*color).astype(np.uint8)
        contour,_ = cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay,contour,-1,tuple(int(v) for v in color),2)
        ys,xs = np.where(mask)
        label = f"{CAT_NAMES[p['category_id']]} {p['score']:.2f}"
        point = int(xs.min()),max(18,int(ys.min()))
        cv2.putText(overlay,label,point,cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),4,cv2.LINE_AA)
        cv2.putText(overlay,label,point,cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),1,cv2.LINE_AA)
    fig,axes = plt.subplots(1,2,figsize=(11,6))
    for ax,graphic,title in zip(axes,(arr,overlay),('Foto baru','Mask hasil inferensi')):
        ax.imshow(graphic)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{model_name} · ambang {threshold:.2f} · {len(kept)} mask')
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if kept:
        print(pd.DataFrame([{'kelas':CAT_NAMES[c],'jumlah':v} for c,v in Counter(p['category_id'] for p in kept).items()]).to_string(index=False))

def ap_satu_kelas(model_name, category, split='original'):
    '''Tampilkan AP mask per kelas dari evaluasi tersimpan.'''
    key,scope = (f'{model_name}_valid','kelas_ada_di_train') if split=='validation' else (
        f'{model_name}_{split}_test','semua_kelas')
    file = METRICS/key/f'AP_per_class_{scope}.csv'
    if not file.is_file():
        print('Tabel AP belum tersedia untuk pilihan ini.')
        return None
    data = pd.read_csv(file)
    row = data.loc[data['class']==category]
    if row.empty or pd.isna(row.iloc[0]['mask_AP50_95']):
        print(f'{category} tidak memiliki ground truth pada set ini.')
        return None
    result = float(row.iloc[0]['mask_AP50_95'])*100
    print(f'{model_name} · {category} · {split} · mask AP50–95 {result:.2f}%')
    return result

def run_live_demo(photo, model_name=DEMO_MODEL, threshold=DEMO_THRESHOLD):
    arr,predictions = predict_demo(photo,model_name)
    render_demo(arr,predictions,threshold,model_name)

if DEMO_PATH is not None:
    run_live_demo(DEMO_PATH, DEMO_MODEL, DEMO_THRESHOLD)
else:
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output
    except ImportError:
        print('Isi DEMO_PATH dengan path foto baru, lalu jalankan ulang cell ini.')
    else:
        uploader = widgets.FileUpload(accept='image/jpeg,image/png',multiple=False,description='Pilih foto')
        choices = [(name,name) for name,_,_ in BASELINES]
        demo_pick = widgets.Dropdown(options=choices,value=DEMO_MODEL,description='Model')
        cutoff = widgets.FloatSlider(value=DEMO_THRESHOLD,min=0.01,max=0.95,step=0.01,
                                     description='Ambang',continuous_update=False)
        execute_demo = widgets.Button(description='Jalankan inferensi',button_style='primary')
        canvas = widgets.Output()
        demo_state = {}

        def redraw_demo(_=None):
            if not demo_state:
                return
            with canvas:
                clear_output(wait=True)
                render_demo(demo_state['photo'],demo_state['predictions'],cutoff.value,demo_state['model'])

        def infer_uploaded(_):
            with canvas:
                clear_output(wait=True)
                items = list(uploader.value.values()) if isinstance(uploader.value,dict) else list(uploader.value)
                if not items:
                    print('Pilih satu foto JPG atau PNG yang baru diambil.')
                    return
                try:
                    arr,pred = predict_demo(items[0]['content'],demo_pick.value)
                except (ValueError,OSError,RuntimeError) as error:
                    print(f'Inferensi gagal. {error}')
                    return
                demo_state.update(photo=arr,predictions=pred,model=demo_pick.value)
            redraw_demo()

        ap_class = widgets.Dropdown(options=sorted(CAT_NAMES[c] for c in CAT_IDS),value='chair',description='Kelas')
        ap_split = widgets.Dropdown(options=[('Test asli','original'),('Test HP','hp'),('Validation','validation')],
                                    value='original',description='Data uji')
        ap_button = widgets.Button(description='Tampilkan AP')
        ap_output = widgets.Output()

        def report_ap(_):
            with ap_output:
                clear_output(wait=True)
                ap_satu_kelas(demo_pick.value,ap_class.value,ap_split.value)

        cutoff.observe(redraw_demo,names='value')
        execute_demo.on_click(infer_uploaded)
        ap_button.on_click(report_ap)
        display(widgets.VBox([uploader,demo_pick,cutoff,execute_demo,canvas,ap_class,ap_split,ap_button,ap_output]))
        print('Pada sesi aktif, unggah foto baru dan klik Jalankan inferensi. '
              'Gunakan Ambang untuk mengubah mask yang ditampilkan, atau Tampilkan AP untuk melihat satu kelas.')


In [ ]:
# Uji teknis pembacaan bobot dan orientasi dimensi dengan gambar sintetis.
test_stream=io.BytesIO()
Image.new('RGB',(128,96),(135,152,168)).save(test_stream,format='PNG')
test_rgb,test_prediction=predict_demo(test_stream.getvalue(),'YOLO11s')
assert test_rgb.shape==(96,128,3) and isinstance(test_prediction,list)
print('Uji teknis inferensi pada gambar sintetis berhasil.')
ap_satu_kelas('YOLO11s','chair','original')
